# Sales Analyst 1.5B v2 Continual Fine-Tune (QLoRA)

## What this notebook does

Unlike `finetune_1.5b.ipynb` and `finetune_3b.ipynb` (which fine-tune the raw
Hugging Face base model from scratch on the full ~1000-pair v1 dataset),
this notebook does **continual fine-tuning**: it starts from the
*already-trained v1 1.5B adapter* (`models/adapters/1.5b/`) and further
trains only on the small v2 "delta" dataset (`data/v2/delta.jsonl`, ~200
pairs covering just the new `forecast_up`/`cohort_question` signal types
introduced in salestools v2.0.0). This teaches the model the new v2 API
surface without re-paying the cost of full v1 training, and — because it
never re-trains on v1 data — avoids the risk of a mixed-dataset retrain
accidentally degrading v1 behavior (catastrophic forgetting). Correspondingly
it uses different hyperparameters suited to a much smaller dataset: 5 epochs
instead of 3, and a lower learning rate (1e-4 vs 2e-4), since it's refining
an already-competent model rather than starting cold.

**Prerequisite**: you must have already run `finetune_1.5b.ipynb` end to
end (fine-tune → export → eval) and have that run's downloaded
`adapter_1.5b.zip` on hand — CELL 4 below will prompt you to upload it,
since the trained v1 adapter is a local-only artifact (`models/adapters/`
is gitignored) that can't be pulled in via the repo clone the way training
data can.

## Run this in Google Colab

This notebook is designed to run in **Google Colab**, not locally — it
depends on a Colab GPU runtime and installs GPU-specific packages
(`unsloth`, and `bitsandbytes` via its extras) that assume a CUDA
environment. To run it:

1. Open this notebook in Colab.
2. Runtime → Change runtime type → GPU (L4 is plenty — this is a short,
   small-dataset run).
3. Run cells top-to-bottom, in order — later cells depend on state created
   by earlier ones (`cfg`, `model`, `dataset`, `trainer`, ...).
4. After CELL 1 (dependency install) finishes, do **Runtime → Restart
   session**, then continue from CELL 2 — this is required so the
   freshly-installed packages are picked up. Don't re-run CELL 1 after
   restarting.
5. At CELL 4, have `adapter_1.5b.zip` (from your `finetune_1.5b.ipynb` run)
   ready to upload when prompted.

**Base model**: `Qwen/Qwen2.5-Coder-1.5B-Instruct` (loaded via the v1 adapter)  
**Hardware**: Colab Pro, L4 GPU (~10-15 min — small dataset)  
**Config**: `training/config/lora_1.5b-v2.yaml`  
**Starting checkpoint**: `models/adapters/1.5b/` (uploaded in CELL 4)  
**Data**: `data/v2/delta.jsonl` (generated by CELL 5 if it doesn't already exist)  
**Output**: `models/adapters/1.5b-v2/` (LoRA adapter, downloaded as a zip by the last cell)

### CELL 1: Install dependencies

Installs `unsloth` plus small extras (`sentencepiece`, `pyyaml`).
Deliberately does **not** pin `peft`/`transformers`/`trl`/`bitsandbytes`/
`accelerate`/`datasets` versions — an earlier version of this notebook
pinned them, which conflicted with current `unsloth-zoo` releases; letting
unsloth manage those versions avoids that. Also installs `salestools`' own
declared dependencies (`pandas`, `statsmodels`, `scikit-learn`,
`matplotlib`) since CELL 5 imports `salestools` transitively (via the data
generator's sandboxed verifier subprocess) if `data/v2/delta.jsonl` needs
regenerating. Also sets a CUDA allocator flag
(`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`) to reduce GPU memory
fragmentation during training.

**After this cell finishes, restart the Colab runtime** (Runtime → Restart
session) so the newly installed packages load cleanly, then continue from
CELL 2 — don't re-run this cell after restarting.

In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────────
import os

# Install fine-tuning dependencies
# Let unsloth manage ALL ML library versions (peft, transformers, trl, accelerate, datasets, bitsandbytes)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q sentencepiece pyyaml

# salestools' own declared dependencies (see pyproject.toml) — installed explicitly rather
# than relying on Colab's default image happening to include them, since CELL 5 imports
# salestools transitively (via the data generator's sandboxed verifier subprocess).
!pip install -q "pandas>=2.0" "statsmodels>=0.14" "scikit-learn>=1.4" "matplotlib>=3.8"

# Prevent CUDA OOM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### CELL 2: Imports

Imports `unsloth` first — it must be imported before
`transformers`/`trl`/`peft` so its performance patches apply. Then imports
the standard libraries used throughout the notebook (`json`, `subprocess`,
`shutil`, `time`, `zipfile`, `pathlib.Path`, `yaml`, `torch`) plus the
Hugging Face/TRL/Unsloth classes used later (`Dataset`, `SFTTrainer`,
`TrainingArguments`, `FastLanguageModel`), plus Colab's `files` helper
(used in CELL 4 to upload the v1 adapter, and in CELL 11 to download the
finished v2 adapter). Finally, checks and prints whether a CUDA GPU is
available — if this prints `CUDA available: False`, stop and fix your
Colab runtime type (Runtime → Change runtime type → GPU) before
continuing.

In [ ]:
# ── CELL 2: Imports ──────────────────────────────────────────────────────────
import unsloth  # must be first — patches trl/transformers/peft before they load

import json, os, subprocess, shutil, time, zipfile
from pathlib import Path
import yaml
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from google.colab import files

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### CELL 3: Paths and config

Defines all the filesystem paths the rest of the notebook uses (repo root,
config YAML, system prompt, delta training data, v1 adapter path, v2
adapter output directory) relative to `/content/salestools-analyst` —
Colab's local (ephemeral) disk, not Google Drive.

If the config file isn't already present, it (re)clones this GitHub repo
into that path — including wiping out any stale/incomplete directory left
over from a previous failed attempt, so a half-finished clone can't
silently make the rest of the notebook fail. It then loads
`training/config/lora_1.5b-v2.yaml` (hyperparameters, LoRA settings) into
the `cfg` dict and reads the system prompt text.

Note this clone gets you the *code* (including the data generator, so
CELL 5 can regenerate `data/v2/delta.jsonl` if needed) but **not** the v1
adapter itself — `models/adapters/` is gitignored, so that has to come from
your local machine via the upload in CELL 4.

In [ ]:
# ── CELL 3: Paths and config ─────────────────────────────────────────────────
REPO_ROOT   = Path("/content/salestools-analyst")
CONFIG_PATH = REPO_ROOT / "training/config/lora_1.5b-v2.yaml"
PROMPT_PATH = REPO_ROOT / "training/config/system_prompt.txt"
DELTA_DATA  = REPO_ROOT / "data/v2/delta.jsonl"
V1_ADAPTER  = REPO_ROOT / "models/adapters/1.5b"      # starting checkpoint (uploaded in CELL 4)
ADAPTER_OUT = REPO_ROOT / "models/adapters/1.5b-v2"   # output

# Clone if config file not present (handles empty-dir edge case)
if not CONFIG_PATH.exists():
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", "https://github.com/sandeepkesarkar/salestools-analyst.git", str(REPO_ROOT)], check=True)

ADAPTER_OUT.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
SYSTEM_PROMPT = Path(PROMPT_PATH).read_text().strip()

print("Config:", cfg)
print(f"System prompt: {SYSTEM_PROMPT[:80]}...")

### CELL 4: Provide the v1 adapter (upload if missing)

This is the one cell with no equivalent in `finetune_1.5b.ipynb` or
`finetune_3b.ipynb`. Those notebooks train from the raw base model, so
everything they need comes from the git clone or from a regenerated
dataset. This notebook instead needs an *already fine-tuned* v1 adapter
as its starting point — and since `models/adapters/` is gitignored (large
binary files), there's no way to clone or regenerate it. If
`V1_ADAPTER/adapter_config.json` isn't already present (it won't be, on a
fresh clone), this cell prompts you to upload `adapter_1.5b.zip` — the
same file `finetune_1.5b.ipynb`'s last cell downloaded — and unzips it
into place.

**Note if using Runtime → Run all**: execution pauses here waiting for the
upload widget — that's expected, not a hang. Watch for the file picker.

In [ ]:
# ── CELL 4: Provide the v1 adapter (upload if missing) ───────────────────────
if not (V1_ADAPTER / "adapter_config.json").exists():
    print(f"v1 adapter not found at {V1_ADAPTER}.")
    print("Upload your locally-exported adapter_1.5b.zip (from finetune_1.5b.ipynb) below:")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    V1_ADAPTER.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(V1_ADAPTER)
    print(f"v1 adapter extracted to {V1_ADAPTER}")
else:
    print(f"v1 adapter already present at {V1_ADAPTER}, skipping upload.")

### CELL 5: Generate v2 delta training data (skip if exists)

If `data/v2/delta.jsonl` doesn't already exist in the cloned repo, runs
`data/generator/generate.py` with `--delta-from 1.0.0` to synthesize ~200
verified pairs covering only the signal types added since v1
(`forecast_up`, `cohort_question`) — the same command used to regenerate
this dataset locally. Same sandboxed verification as the v1 generator (see
`finetune_1.5b.ipynb` CELL 4's docs for how that works).

In [ ]:
# ── CELL 5: Generate v2 delta training data (skip if exists) ────────────────
if not DELTA_DATA.exists():
    subprocess.run(
        ["python", "data/generator/generate.py", "--salestools-version", "2.0.0", "--delta-from", "1.0.0",
         "--seed", "42", "--count", "200", "--output", str(DELTA_DATA)],
        cwd=str(REPO_ROOT), check=True
    )
else:
    print("delta.jsonl already exists, skipping generation.")

### CELL 6: Load training data

Reads `delta.jsonl` line by line, keeps only pairs marked `verified: true`,
and reformats each `(question, code)` pair into a single ChatML-style
training string, same format as `finetune_1.5b.ipynb` CELL 5 (system prompt
+ user question + assistant code, wrapped in `<|im_start|>`/`<|im_end|>`
tags). Wraps the resulting list of strings in a Hugging Face `Dataset`
object (`dataset`), which is what the trainer consumes in CELL 8/9.

In [ ]:
# ── CELL 6: Load training data ───────────────────────────────────────────────
CHATML_TEMPLATE = (
    "<|im_start|>system\n{system}<|im_end|>\n"
    "<|im_start|>user\n{user}<|im_end|>\n"
    "<|im_start|>assistant\n{assistant}<|im_end|>"
)

records = []
with open(DELTA_DATA) as f:
    for line in f:
        pair = json.loads(line.strip())
        if not pair.get("verified", False):
            continue
        records.append({"text": CHATML_TEMPLATE.format(
            system=SYSTEM_PROMPT,
            user=pair["question"],
            assistant=pair["code"],
        )})

dataset = Dataset.from_list(records)
print(f"Loaded {len(dataset)} verified v2 delta pairs")
print("Sample:\n", dataset[0]["text"][:300])

### CELL 7: Load the v1 adapter (continue training it directly)

Loads the v1 fine-tuned checkpoint via Unsloth's
`FastLanguageModel.from_pretrained(model_name=str(V1_ADAPTER), ...)` —
passing a saved LoRA adapter directory (rather than a plain Hugging Face
model name) is a supported Unsloth/PEFT pattern for continuing training
from a previous checkpoint: it reads the base model reference out of the
adapter's own `adapter_config.json`, loads that base model in 4-bit, and
attaches the v1 adapter.

**Important**: PEFT defaults adapter loading to `is_trainable=False`
(frozen, inference-only) — passing `is_trainable=True` here is what makes
this a *continuation* of v1 training rather than a frozen starting point.
This notebook deliberately does **not** call `get_peft_model` again to
stack a second, independent adapter on top (an earlier version did) —
`get_peft_model`/`save_pretrained` only ever persist the *currently
active* adapter's deltas, so a second, separately-added adapter risks
`models/adapters/1.5b-v2/` ending up encoding only the v2-specific delta,
with no representation of v1's learned behavior at all. Since
`export.sh` merges whatever's in that directory straight onto the *raw*
base model (bypassing v1 entirely), that would silently produce a model
that knows the new v2 patterns but has forgotten v1 — the opposite of
what continual fine-tuning is supposed to guarantee. Continuing to train
the *same* adapter avoids the ambiguity: the single saved adapter always
represents "v1, further updated by v2 data" as one coherent set of deltas
relative to the raw base model, which is exactly what `export.sh`'s
existing merge step expects.

**Whether Unsloth's wrapper actually accepts/forwards `is_trainable`
couldn't be confirmed without executing this cell** (the equivalent
assumption on `finetune_3b.ipynb` — that periodic checkpointing would just
work — turned out wrong only once actually run, crashing with a
`PicklingError`). So rather than just printing
`model.print_trainable_parameters()` and asking you to read it, this cell
computes the trainable parameter count directly and **hard-asserts it's
nonzero** — if `is_trainable` silently didn't take effect, CELL 8/9 would
otherwise train for ~10-15 min against a frozen model and produce a v2
adapter indistinguishable from v1, with no error at all, only surfacing
later (expensively) at T044's lifecycle eval. This assertion turns that
silent failure mode into an immediate, loud one.

In [ ]:
# ── CELL 7: Load the v1 adapter (continue training it directly) ─────────────
print(f"Loading v1 adapter from {V1_ADAPTER} (base model resolved from its adapter_config.json)...")
_t0 = time.time()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(V1_ADAPTER),
    max_seq_length=cfg["training"]["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
    is_trainable=True,   # continue training this adapter (PEFT defaults to frozen/inference-only)
)
print(f"v1 adapter loaded in {time.time() - _t0:.0f}s.")
model.print_trainable_parameters()

# Hard guard: if is_trainable silently didn't take effect, training would run for
# ~10-15 min against a frozen model with no error, producing a v2 adapter indistinguishable
# from v1 — fail loudly now instead of discovering this later at the lifecycle eval.
_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {_trainable:,}")
assert _trainable > 0, (
    "No trainable parameters after loading the v1 adapter with is_trainable=True — "
    "it may have loaded frozen (check Unsloth/PEFT version compatibility with this kwarg) "
    "before proceeding to training."
)

### CELL 8: SFTTrainer setup

Builds a TRL `SFTTrainer`, same pattern as the other two notebooks.
Hyperparameters come from `cfg["training"]` in `lora_1.5b-v2.yaml` — tuned
for this much smaller (~200-pair) dataset: 5 epochs instead of 3, a lower
learning rate (1e-4 vs 2e-4), and a linear (not cosine) scheduler, since
this is refining an already-competent model rather than training from
scratch. `save_strategy: "no"` — an earlier version of this notebook used
periodic checkpointing here, but that's a known Unsloth+TRL compatibility
crash (`PicklingError` on `trl`'s `SFTConfig` at checkpoint-save time,
confirmed when it happened on `finetune_3b.ipynb`'s first real run) —
`"no"` avoids it entirely, same as both other notebooks. Doesn't start
training yet — just prepares everything so CELL 9 can.

In [ ]:
# ── CELL 8: SFTTrainer setup ──────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=cfg["training"]["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=cfg["training"]["per_device_train_batch_size"],
        gradient_accumulation_steps=cfg["training"]["gradient_accumulation_steps"],
        warmup_ratio=cfg["training"]["warmup_ratio"],
        num_train_epochs=cfg["training"]["num_train_epochs"],
        learning_rate=cfg["training"]["learning_rate"],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=cfg["training"]["logging_steps"],
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type=cfg["training"]["lr_scheduler_type"],
        seed=cfg["training"]["seed"],
        output_dir=str(ADAPTER_OUT / "checkpoints"),
        save_strategy=cfg["training"]["save_strategy"],
        save_total_limit=2,
        report_to="none",
    ),
)
print("Trainer ready.")

### CELL 9: Train

Runs the fine-tuning loop (`trainer.train()`) — much shorter than the
other two notebooks (~10-15 min) given the small (~200-pair) dataset and
L4-class hardware being plenty here. Because CELL 7 continues training the
same v1 adapter (not a fresh one), this run refines those existing weights
rather than starting from random initialization. Logs training loss every
`logging_steps` steps (from config) as it goes. Prints total wall-clock
runtime when done.

In [ ]:
# ── CELL 9: Train ─────────────────────────────────────────────────────────────
print(f"Starting continual fine-tune ({cfg['training']['num_train_epochs']} epochs, "
      f"logging every {cfg['training']['logging_steps']} steps)...")
trainer_stats = trainer.train()
print(f"Training complete. Runtime: {trainer_stats.metrics.get('train_runtime', 0):.0f}s")

### CELL 10: Save v2 adapter

Saves the adapter weights and tokenizer to `ADAPTER_OUT`
(`models/adapters/1.5b-v2/`) using PEFT's `save_pretrained`. Because CELL
7-9 continued training the *same* adapter loaded from `V1_ADAPTER` (rather
than stacking a fresh one), this save captures a single, complete set of
deltas representing "v1, further updated by v2 training" relative to the
raw base model — not a merged copy of the base model itself. This is what
makes it safe to merge directly onto the raw base model in `export.sh`
without losing v1's learned behavior. Lists the files written so you can
sanity-check the save succeeded.

In [ ]:
# ── CELL 10: Save v2 adapter ──────────────────────────────────────────────────
model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
print(f"v2 adapter saved to {ADAPTER_OUT}")
print("Files:", list(ADAPTER_OUT.glob("*")))

### CELL 11: Download adapter

Removes `ADAPTER_OUT/checkpoints/` first if present (defensive cleanup —
should already be empty given `save_strategy: "no"`), then zips the saved
adapter directory and triggers a browser download via Colab's
`google.colab.files.download` — this is how you get the trained adapter
off the (ephemeral) Colab VM and onto your local machine, since anything
under `/content` is deleted when the Colab runtime disconnects/recycles.
See the "Next Steps" cell below for what to do with it once downloaded.

In [ ]:
# ── CELL 11: Download adapter ─────────────────────────────────────────────────
shutil.rmtree(ADAPTER_OUT / "checkpoints", ignore_errors=True)

shutil.make_archive("/content/adapter_1.5b-v2", "zip", str(ADAPTER_OUT))
files.download("/content/adapter_1.5b-v2.zip")

## Next Steps

1. The last cell downloads `adapter_1.5b-v2.zip` to your machine — unzip it into `models/adapters/1.5b-v2/`
2. Run `bash training/export.sh` (set `MODEL_SIZE=1.5b-v2 ADAPTER_PATH=models/adapters/1.5b-v2/`)
3. Verify v1 behavior wasn't lost: `ollama run sales-analyst-1.5b-v2 "Is my trend going up?"`
4. Verify v2 behavior: `ollama run sales-analyst-1.5b-v2 "Forecast next quarter's revenue"`
5. Lifecycle eval (T044) — v1 held-out set, to confirm v1 behavior is preserved: `python eval/run_eval.py --model sales-analyst-1.5b-v2 --held-out data/v1/held_out.jsonl`
6. Lifecycle eval — v2 delta set, to confirm the new v2 patterns were learned: `python eval/run_eval.py --model sales-analyst-1.5b-v2 --held-out data/v2/delta.jsonl`
7. Compare against the v1-only model: `python eval/compare.py eval/reports/sales-analyst-1.5b-*.json eval/reports/sales-analyst-1.5b-v2-*.json`